In [100]:

from pydantic import BaseModel

# what issue looks like 
def insert_patient_data(name:str,age:int):
    """
    This function will accept age as a string also this means no type checking and validation
    """
    print(name)
    print(age)
    print("Insrted sUCcessfully")

insert_patient_data("feroz","thiry two")



feroz
thiry two
Insrted sUCcessfully


In [101]:
# how we can solve issue 
def insert_name_and_age(name:str,age:int):
    if type(name)==str and type(age) == int:
        print(name)
        print(age)
        print("inserted succesfully ")
    else:
        raise TypeError("Incorrect datatype")

insert_name_and_age("fairoz",32)




fairoz
32
inserted succesfully 


In [102]:
# how pydantic solves it

class Patient(BaseModel):
      name:str
      age:int

def insert_patient_data_pydantic(schema:Patient):
    print(schema.name)
    print(schema.age)
    print("Insertion Success")

In [103]:
pateint_info={'name':"h12ssain","age":"32"}
patient_1=Patient(**pateint_info)
insert_patient_data_pydantic(patient_1)


h12ssain
32
Insertion Success


In [104]:
# Intelliegnt enough to figure convert the age string into the number
type(patient_1.age)

int

# **Lets take things Ahead**

In [105]:
from typing import List, Annotated,Optional,Dict
from pydantic import BaseModel,EmailStr,AnyUrl,Field

In [106]:
class Patient(BaseModel):
    name:str
    age:int
    email:EmailStr
    linkedIn:AnyUrl
    height:float
    weight:float
    married:bool

In [107]:
def update_patient_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print("success")

In [108]:
patient_data={"name":"fairoz","age":32,"email":"fairoz@gmail.com","linkedIn":"https://www.linkedin.com/something","height":5.6,"weight":65,"married":True}

In [109]:
patient_data1=Patient(**patient_data)
update_patient_data(patient_data1)

fairoz
32
fairoz@gmail.com
success


# LETS NOW CHECK HOW TO SET 
* DEFAULT VALUES
* SOME Constrains on inputs etc 
 

In [110]:
class Patient(BaseModel):
        name:str=Field(max_length=10)
        age:int
        email:EmailStr
        linkedIn:AnyUrl
        height:float
        weight:float
        married:bool
        allergies:Optional[List[str]] = None # this shows this value is optional and the default value is None
      


In [111]:
patient_data={"name":"faiz","age":32,"email":"fairoz@gmail.com","linkedIn":"https://www.linkedin.com/something","height":5.6,"weight":65,"married":True,'allergies':['pollen','dust']}

In [112]:
def update_patient_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.allergies)
    print("success")

In [113]:
patient_data1=Patient(**patient_data)
update_patient_data(patient_data1)

faiz
32
fairoz@gmail.com
['pollen', 'dust']
success


# **More Complexity**

In [114]:
class Patient(BaseModel):
        name:Annotated[str,Field(max_length=50,title='Name of the Patient',description="Gve tha name of patient in not more than 50 characters ",examples=['Fairoz','Ahmad','Ali'])]
        age:int=Field(gt=0,lt=100)
        email:EmailStr
        linkedIn:AnyUrl
        height:float=Field(gt=0)
        weight:Annotated[float,Field(gt=0,strict=True)]  # strict means follow strict datatype dont convert string to numbers
        married:Annotated[bool,Field(default=False,description="Is the patient married (True or False)")]
        contact:Dict[str,str]
        allergies:Annotated[Optional[List[str]],Field(default=None,max_length=5)] # this shows this value is optional and the default value is None
      


In [115]:
patient_data={"name":"faiz","age":32,"email":"fairoz@gmail.com","linkedIn":"https://www.linkedin.com/something","height":5.6,"weight":65,"married":True,'contact':{"email":"helpful@123"},'allergies':['pollen','dust']}

In [116]:
pt_dt_1=Patient(**patient_data)

In [117]:
def update_patient_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.allergies)
    print(patient.contact)
    print("success")

In [118]:
update_patient_data(pt_dt_1)

faiz
32
fairoz@gmail.com
['pollen', 'dust']
{'email': 'helpful@123'}
success


# **Feild Validator**

Used to check a particular feild here we will check if the email of patient is from hdfc or icici if not we wont allow them to create an account
Validator does two thing 
* Custom Data Validation 
* Transformation

In [119]:
from pydantic import field_validator
class Patient(BaseModel):
    name:str
    age:int
    email:EmailStr

    # Email checking 
    @field_validator('email')
    @classmethod
    def email_validator(cls,value):
        valid_domains=['hdfc.com','icici.com']
        domain_name=value.split("@")[-1]
        if domain_name not in valid_domains:
            raise ValueError('Not a Valid Domain')
        return value
    
    # Username Transformation
    @field_validator('name')
    @classmethod
    def transform_name(cls,value):
        return value.upper()


In [120]:
def update_data(patient:Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)


In [121]:
data={'name':"fairoz","age":32,'email':"fairoz@icici.com"}
pd_1=Patient(**data)
update_data(pd_1)

FAIROZ
32
fairoz@icici.com


# **Model Validator**


Suppose we want to check multiple things example if age > 60 the contact should have emergency contact otherwise it should not allow creation


In [122]:
import warnings
warnings.filterwarnings('ignore')

In [123]:
from pydantic import model_validator
class Patient(BaseModel):
    name:str
    age:int
    contact:Dict[str,str] 
    @model_validator(mode="after")
    def validate_emergency_contact(cls,model):
        if model.age>60 and 'emergency_contact' not in model.contact:
            raise ValueError('Please Provide an emergency contact ')
        return model

In [124]:
data={'name':"fairoz","age":91,'contact':{"emergency_contact":"100"}}
pd_1=Patient(**data)

In [125]:
def update_details(schema:Patient):
    print(schema.name)
    print(schema.age)
    print(schema.contact)
update_details(pd_1)

fairoz
91
{'emergency_contact': '100'}


# **Computed Feild**

In [129]:
from pydantic import computed_field
class New_patient(BaseModel):
    name:str
    age:int
    weight:float
    height:float


    @computed_field
    @property
    def bmi(self)->float:
        bmi=round(self.weight/(self.height**2),2)
        return bmi




In [130]:
data={"name":"waris",'age':22,"weight":41.6,'height':4.6}
dt_1=New_patient(**data)


In [133]:
def update_data(pat:New_patient):
    print(pat.name)
    print(pat.age)
    print(pat.height)
    print(pat.weight)
    print('BMI ',pat.bmi)

In [134]:
update_data(dt_1)

waris
22
4.6
41.6
BMI  1.97
